# Local Validation

This notebook is used to validate a local version of the Animl deployment of small-animal-classififer against an output JSON produced by running a cloned version of the small-animal-classififer github repository. If you would like to use a different set of images or reproduce the JSON output file for the provided images, please follow the instructions here: https://github.com/agentmorris/small-animal-classifier/.

## Setting Up Docker

```bash
cd models/small-animal-classifier
docker buildx build --platform linux/amd64 -t small-animal-classifier .
docker run -p 8080:8080 small-animal-classifier
```

This starts up a FastAPI server at port `8080` on your local machine. It can be reached at `http://localhost:8080`.

To check the model loaded correctly and the server is working:
`curl http://localhost:8080/ping`

## Imports

In [2]:
from pathlib import Path
from PIL import Image
import base64
import requests
import json

## Prepare Images for Inferences

In [3]:
image_dir = Path("./images")
images_b64 = {}
for image_path in image_dir.glob("*.jpg"):
    with open(image_path, "rb") as image_bytes:
        image_bytes = base64.b64encode(image_bytes.read()).decode("utf-8")
        images_b64[image_path.name] = image_bytes

print(f"loaded and encoded {len(images_b64)} images from {image_dir.resolve()}")

loaded and encoded 6 images from /Users/jesseleung/Projects/tnc-projects/animl/animl-ml/models/small-animal-classifier/validation/images


## Run Inference on Docker Model

In [8]:
inference_res = {}
for image_name, image_b64 in images_b64.items():
    res = requests.post(
        "http://localhost:8080/invocations",
        json={"image": image_b64}
    )
    inference_res[image_name] = res.json()

## Compare to Sample Output

In [10]:
with open("sample_output.json") as f:
    sample_data = json.load(f)
sample_inference = sample_data["images"]
class_names = sample_data["classification_categories"]

for image_name, res in inference_res.items():
    sample_res = next(sample for sample in sample_inference if sample["file"] == image_name)
    if sample_res is None:
        raise Exception("Did not compare the same files")
        
    sample_classifications = sample_res["detections"][0]["classifications"]
    top_3 = sorted(res.items(), key=lambda x: x[1], reverse=True)[:3]

    for classification, confidence in top_3:
        match = next(c for c in sample_classifications if class_names[c[0]] == classification)
        if match is None:
            raise Exception("Top 3 classifications did not match")
        if abs(match[1] - confidence) >= 0.00005:
            print(match[1], confidence)
            raise Exception(f"Top 3 confidence scores did not match. Expected: {match[1]}, got: {confidence}")
        print(f"Match! Expected: {match[1]}, got: {round(confidence, 4)}, rounded from: {confidence}") 

print("Classifications and confidence scores matched sample data")
    
    

Match! Expected: 0.542, got: 0.542, rounded from: 0.5419602394104004
Match! Expected: 0.0665, got: 0.0665, rounded from: 0.06652276962995529
Match! Expected: 0.038, got: 0.038, rounded from: 0.03799259290099144
Match! Expected: 0.8005, got: 0.8005, rounded from: 0.8005000352859497
Match! Expected: 0.0282, got: 0.0282, rounded from: 0.028188759461045265
Match! Expected: 0.0151, got: 0.0151, rounded from: 0.015063728205859661
Match! Expected: 0.8556, got: 0.8556, rounded from: 0.8556024432182312
Match! Expected: 0.0203, got: 0.0203, rounded from: 0.020347317680716515
Match! Expected: 0.0172, got: 0.0172, rounded from: 0.01724953204393387
Match! Expected: 0.9008, got: 0.9008, rounded from: 0.9007808566093445
Match! Expected: 0.0124, got: 0.0124, rounded from: 0.0124015212059021
Match! Expected: 0.0109, got: 0.0109, rounded from: 0.01088184304535389
Match! Expected: 0.7737, got: 0.7737, rounded from: 0.773716926574707
Match! Expected: 0.033, got: 0.033, rounded from: 0.03297265246510506
Ma

## (Optional) Compare Against Random Sample From California Small Animals dataset

To double check that the deployment is accurate to Dan Morris's repository, download a random sample of images from the [California Small Animals dataset](https://lila.science/datasets/california-small-animals/). The `download_sample.py` script is set up to download 100 images from the first 9000 images. After downloading the sample, you will need to run Dan Morris's model according to Github on the images to get a ground truth output.

In [11]:
downloaded_image_dir = Path("./images/downloaded")
downloaded_images_b64 = {}
for image_path in downloaded_image_dir.glob("*.jpg"):
    with open(image_path, "rb") as image_bytes:
        image_bytes = base64.b64encode(image_bytes.read()).decode("utf-8")
        downloaded_images_b64[image_path.name] = image_bytes

print(f"loaded and encoded {len(downloaded_images_b64)} images from {downloaded_image_dir.resolve()}")

loaded and encoded 100 images from /Users/jesseleung/Projects/tnc-projects/animl/animl-ml/models/small-animal-classifier/validation/images/downloaded


In [12]:
downloaded_inference_res = {}
completed = 0
for image_name, image_b64 in downloaded_images_b64.items():
    res = requests.post(
        "http://localhost:8080/invocations",
        json={"image": image_b64}
    )
    downloaded_inference_res[image_name] = res.json()
    completed += 1
    if completed % 10 == 0:
        print(f"completed {completed} images")

completed 10 images
completed 20 images
completed 30 images
completed 40 images
completed 50 images
completed 60 images
completed 70 images
completed 80 images
completed 90 images
completed 100 images


In [18]:
with open("./downloaded_output.json") as f:
    downloaded_sample_data = json.load(f)
downloaded_sample_inference = downloaded_sample_data["images"]
downloaded_class_names = downloaded_sample_data["classification_categories"]

for image_name, res in downloaded_inference_res.items():
    sample_res = next(sample for sample in downloaded_sample_inference if sample["file"] == image_name)
    if sample_res is None:
        raise Exception("Did not compare the same files")
        
    sample_classifications = sample_res["detections"][0]["classifications"]
    top_3 = sorted(res.items(), key=lambda x: x[1], reverse=True)[:3]

    for classification, confidence in top_3:
        match = next(c for c in sample_classifications if downloaded_class_names[c[0]] == classification)
        if match is None:
            raise Exception("Top 3 classifications did not match")
        if abs(match[1] - confidence) >= 0.00005:
            print(match[1], confidence)
            raise Exception(f"Top 3 confidence scores did not match. Expected: {match[1]}, got: {confidence}")
        print(f"Match! Expected: {match[1]}, got: {round(confidence, 4)}, rounded from: {confidence}") 

print("Classifications and confidence scores matched sample data")


Match! Expected: 0.6141, got: 0.6141, rounded from: 0.6140718460083008
Match! Expected: 0.0549, got: 0.0549, rounded from: 0.05494874343276024
Match! Expected: 0.0316, got: 0.0316, rounded from: 0.03161988407373428
Match! Expected: 0.8977, got: 0.8977, rounded from: 0.8976629972457886
Match! Expected: 0.016, got: 0.016, rounded from: 0.016048170626163483
Match! Expected: 0.0082, got: 0.0082, rounded from: 0.008204235695302486
Match! Expected: 0.4796, got: 0.4796, rounded from: 0.47955021262168884
Match! Expected: 0.1265, got: 0.1265, rounded from: 0.12650255858898163
Match! Expected: 0.0577, got: 0.0577, rounded from: 0.057724080979824066
Match! Expected: 0.6358, got: 0.6358, rounded from: 0.6358239650726318
Match! Expected: 0.0511, got: 0.0511, rounded from: 0.05110526457428932
Match! Expected: 0.0296, got: 0.0296, rounded from: 0.029596002772450447
Match! Expected: 0.6162, got: 0.6162, rounded from: 0.6161515116691589
Match! Expected: 0.0554, got: 0.0554, rounded from: 0.055413652211